# Executed notebook: E06_genie

Exported from Databricks job run `980301348000608` (task `E06_genie`, task run `1027823489205687`).

Result: **SUCCESS** · start 2026-10-06T17:46:32.647000+00:00 · end 2026-10-06T17:51:19.737000+00:00

Run URL: https://fevm-serverless-stable-am1uc2.cloud.databricks.com/?o=7474651880045550#job/755461157363253/run/980301348000608


# E06 · Genie agent on metric views and work orders

**What was built.** Genie agent "Plant Maintenance Agent", defined as code in `src/genie/space_config.py`:
3 metric views + `station_health_current` + `work_orders_current`, text instructions (plant naming, risk bands,
compressed time, where to find what), 4 example SQL queries, 4 sample questions and 10 benchmark questions with
reference SQL. Deployed with `tools/genie_deploy.py`.

**What this notebook proves.** The deployed configuration, and live answers: each benchmark question is asked through
the Genie Conversation API; Genie's generated SQL and the reference SQL are executed back to back and compared.

In [ ]:
%pip install -q "databricks-sdk>=0.81" pg8000
%restart_python

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [ ]:
from _helpers import *  # noqa: F401,F403
import sys, os, time, json
sys.path.append(os.path.abspath("../.."))
from genie import space_config as cfg

## 1 · Deployed configuration

In [ ]:
sp = w.api_client.do("GET", f"/api/2.0/genie/spaces/{GENIE_SPACE_ID}", query={"include_serialized_space": "true"})
ser = json.loads(sp["serialized_space"])
print("title:", sp["title"], "| warehouse:", sp["warehouse_id"])
print("data sources:", [t["identifier"].split(".", 1)[1] for k in ("tables", "metric_views") for t in ser["data_sources"].get(k, [])])
print("sample questions:", [q["question"][0] for q in ser["config"]["sample_questions"]])
print("example SQL questions:", [q["question"][0] for q in ser["instructions"]["example_question_sqls"]])
print("benchmarks:", len(ser["benchmarks"]["questions"]))
print("\ninstructions:\n" + ser["instructions"]["text_instructions"][0]["content"][0])

title: Plant Maintenance Agent | warehouse: fb9bc265e9f4578a
data sources: ['pdm_core.station_health_current', 'pdm_ops.maintenance_metrics', 'pdm_ops.station_risk_metrics', 'pdm_ops.work_order_metrics', 'pdm_ops.work_orders_current']
sample questions: ['How many open work orders do we have by priority?', 'Why is the riskiest station at risk?', 'Which stations need maintenance soon?', 'Which station types fail most often and what is their MTTR?']
example SQL questions: ['How did average risk develop per plant over the last 15 minutes?', 'How many open work orders are there per plant?', 'Which stations are at high risk right now?', 'What is the MTTR and failure count by station type?']
benchmarks: 10

instructions:
You are a maintenance planning assistant for three plants (Plant North = PLT-N, Plant South = PLT-S, Plant East = PLT-E),


## 2 · Live benchmark: Genie answer vs reference SQL

In [ ]:
def norm(rows):
    out = []
    for r in rows:
        vals = set()
        for v in r:
            try:
                vals.add(round(float(v), 2))
            except (TypeError, ValueError):
                vals.add(None if v is None else str(v))
        out.append(vals)
    return out


results = []
for q, ref_sql in cfg.benchmarks(CATALOG):
    t0 = time.time()
    msg = w.genie.start_conversation_and_wait(GENIE_SPACE_ID, q)
    secs = time.time() - t0
    sql = next((a.query.query for a in (msg.attachments or []) if a.query), None)
    text = next((a.text.content for a in (msg.attachments or []) if a.text and a.text.content), None)
    g_rows = [list(r) for r in spark.sql(sql).collect()] if sql else []
    r_rows = [list(r) for r in spark.sql(ref_sql).collect()]
    ok = bool(sql) and len(g_rows) == len(r_rows) and all(any(rr <= gg for gg in norm(g_rows)) for rr in norm(r_rows))
    results.append((("PASS" if ok else "FAIL"), round(secs, 1), q))
    print(f"\n### {'PASS' if ok else 'FAIL'} ({secs:.1f} s): {q}")
    print("Genie SQL:\n" + (sql or "<none>"))
    print("Genie rows:", g_rows[:5])
    print("Reference rows:", r_rows[:5])
    if text:
        print("Genie text:", text[:400])

### PASS (18.0 s): How many stations are at high risk right now?
Genie SQL:
SELECT COUNT(*) AS `high_risk_stations`
FROM `serverless_stable_am1uc2_catalog`.`pdm_core`.`station_health_current`
WHERE `risk_band` = 'HIGH'
Genie rows: [[1]]
Reference rows: [[1]]
Genie text: There are **1** stations at **HIGH** risk right now based on `pdm_core.station_health_current`. This means the current count of stations in the **HIGH** risk band is **1**.

### PASS (17.7 s): How many stations are currently down?
Genie SQL:
SELECT COUNT(*) AS `stations_down`
FROM `serverless_stable_am1uc2_catalog`.`pdm_core`.`station_health_current`
WHERE `risk_band` = 'DOWN'
Genie rows: [[0]]
Reference rows: [[0]]
Genie text: There are **0 stations** currently marked as **DOWN**. Based on the current station status data, **no stations are currently down**.

### PASS (23.4 s): What are the 5 stations with the highest failure probability right now?
Genie SQL:
WITH ranked AS (
  SELECT
    station_id,
    plant_id,
    l

In [ ]:
display(spark.createDataFrame(results, "verdict string, seconds double, question string"))
print(f"{sum(r[0] == 'PASS' for r in results)}/{len(results)} benchmark questions match the reference result")

| verdict | seconds | question |
|---|---|---|
| PASS | 18.0 | How many stations are at high risk right now? |
| PASS | 17.7 | How many stations are currently down? |
| PASS | 23.4 | What are the 5 stations with the highest failure probability right now? |
| PASS | 18.1 | Which station type had the most failures? |
| PASS | 12.8 | What is the mean time to repair in minutes for each plant? |
| PASS | 19.1 | Which failure mode caused the most downtime? |
| PASS | 12.1 | What is the total parts cost per plant? |
| PASS | 19.0 | What share of repairs were preventive for each station type? |
| PASS | 12.3 | How many failures did Plant North have? |
| PASS | 18.0 | List the stations on line PLT-S-B with their current risk band. |


10/10 benchmark questions match the reference result


## 3 · Sample questions (free-form answers)

In [ ]:
for q in cfg.SAMPLE_QUESTIONS:
    msg = w.genie.start_conversation_and_wait(GENIE_SPACE_ID, q)
    sql = next((a.query.query for a in (msg.attachments or []) if a.query), None)
    text = next((a.text.content for a in (msg.attachments or []) if a.text and a.text.content), None)
    print(f"\n### {q}\nGenie text: {text}\nGenie SQL:\n{sql}")
    if sql:
        print("rows:", [list(r) for r in spark.sql(sql).limit(5).collect()])

### Which stations need maintenance soon?
Genie text: **6 stations** currently need maintenance soon: **5 are HIGH risk** and **1 is ELEVATED**.

Examples include:
- **PLT-N-C01** (Plant North) — **99.86%** risk, **HIGH**, driven by **hydraulic_pressure_bar**
- **PLT-S-B01** (Plant South) — **99.28%** risk, **HIGH**, driven by **motor_current_a**
- **PLT-S-D01** (Plant South) — **97.08%** risk, **HIGH**, driven by **cycle_time_s**
- **PLT-S-A01** (Plant South) — **92.76%** risk, **HIGH**, driven by **cycle_time_s**
- **PLT-S-C04** (Plant South) — **70.83%** risk, **HIGH**, driven by **vibration_rms**

Plant South accounts for **4 of the 6** at-risk stations, and the risk levels range from **50.76%** to **99.86%**.
Genie SQL:
SELECT `station_id`, `plant_id`, `line_id`, `station_type`, `criticality`, `risk_band`, `failure_probability` * 100 AS `risk_pct`, `top_signal`, `top_signal_deviation_pct`, `last_reading_ts`
FROM `serverless_stable_am1uc2_catalog`.`pdm_core`.`station_health_current